Program to import Soundwall Entries into JSON parent from Excel worksheets. 

In [1]:
import pandas as pd
import json
import os

def excel_to_json(input_dir, output_dir):
    # Read the Excel file
    excel_file = pd.ExcelFile(input_dir)
    all_data = []

    # Iterate through each sheet in the Excel file
    for sheet_index, sheet_name in enumerate(excel_file.sheet_names):
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_suffix = f"{sheet_index + 1:02d}"  # Ensure unique IDs for each tab
        
        # Create the first icon element from the first row of data
        first_row = df.iloc[0]
        icon_element = {
            "type": "icn",
            "role": "image",
            "alt": first_row.get("alt", ""),
            "id": f"icn{sheet_suffix}",
            "content": first_row.get("content", ""),
            "left": first_row.get("left", ""),
            "top": first_row.get("top", ""),
            "action": "openGroupGhost",
            "target": f"grp{sheet_suffix}"
        }
        all_data.append(icon_element)
        
        # Build the group object
        group = {
            "type": "grp",
            "id": f"grp{sheet_suffix}",
            "style": "grpAdv",
            "left": "4em",
            "top": "2em",
            "width": "56em",
            "height": "25em",
            "visible": "false",
            "children": []
        }
        
        # Iterate through rows to create child objects
        for index, row in df.iterrows():
            if index == 0:
                continue  # Skip the first row which has already been processed as an icon element
            
            # Create the child object by omitting blank fields
            child = {key: value for key, value in row.items() if pd.notna(value) and value != ""}
            
            if child.get('style') == 'words':
                child['children'] = [
                    {
                        "type": "icn",
                        "role": "image",
                        "content": child.get('content'),
                        "left": "-4em",
                        "top": "-1em",
                        "height": "8em",
                        "width": "8em"
                    },
                    {
                        "type": "txt",
                        "style": "vocab",
                        "content": child.get('text'),
                        "left": "-.0em",
                        "top": ".25em"
                    }
                ]
            
            group['children'].append(child)
        
        all_data.append(group)
    
    # Write to JSON file without square brackets
    output_file = os.path.join(output_dir, "output.json")
    with open(output_file, 'w') as json_file:
        for i, item in enumerate(all_data):
            json.dump(item, json_file, indent=4)
            if i < len(all_data) - 1:
                json_file.write(",\n")

def concatenate_text_files(header_file, snippet_file, footer_file, output_file):
    try:
        with open(header_file, 'r') as file:
            header_data = file.read()
    except FileNotFoundError:
        print(f"File not found: {header_file}")
        return

    try:
        with open(snippet_file, 'r') as file:
            snippet_data = file.read()
    except FileNotFoundError:
        print(f"File not found: {snippet_file}")
        return

    try:
        with open(footer_file, 'r') as file:
            footer_data = file.read()
    except FileNotFoundError:
        print(f"File not found: {footer_file}")
        return

    combined_data = header_data + snippet_data + footer_data

    with open(output_file, 'w') as file:
        file.write(combined_data)
    print(f"Combined file written to {output_file}")

# Specify the input and output file paths for Excel to JSON conversion
input_dir = "datacc2.xlsx"
output_dir = ""
excel_to_json(input_dir, output_dir)

# Specify the input and output file paths for concatenation
header_file = "header2.json"
snippet_file = "output.json"
footer_file = "footer.json"
output_file = "../data/soundwallCC2.json"

# Call the function to concatenate the text files
concatenate_text_files(header_file, snippet_file, footer_file, output_file)


Combined file written to ../data/soundwallCC2.json


Now we will concatenate the output with the header and footer to create the entire JSON output